# FFM_3_9 — 在预测任务上微调解码器：冻结的解码器是不是 mesh → HRTF 的瓶颈？

**2026-10-16 建。设计在写代码之前登记（PROJECT_LOG §2C 2026-10-16），运行前又登记了一次修正和补充（同一条下的 "Amendment"）。**
导师的建议：mesh → z → decoder → HRTF 这条路里，解码器是冻结的、回归是在 z 上拟合的；如果让解码器适应预测任务
（损失直接算在实测 HRTF 上），几何的结果会不会变？

## 只动解码器，不动编码器

编码器在预测路径里根本不出现：新受试者没有 HRTF 可编码，ẑ 来自几何。编码器只决定 ridge 的**目标** z。
同时微调编码器会让每个人的目标 z 都变，ridge 要重拟合，之前所有基于 z 的数都不再可比，而且那等于"学一个更容易被几何预测的
latent"，是另一个问题。所以编码器冻结，z_meas 是 FFM_3_8 的。

## 设计（与 log 相同）

- **受试者与折**：FFM_3_8 的 164 个 SONICOM 开发受试者、FFM_3_2_1 的 10 折。**只有开发集**（锁定 36 人已在 10-12 用掉，
  其余 SONICOM 测试受试者没有网格），结果是探索性的；本 notebook 没有测试格，也不读锁定受试者。
- **输入 ẑ**：测试行 = 只在该折训练人上拟合的核 ridge（FFM_3_8 的做法，并逐元素核对 FFM_3_8 的 `dev_oof.npz`）。
  **训练行 = 在该折训练人内部交叉拟合（5 组）**，α 用该折已选的值 —— 这样测试人的信息不会经由训练行的 ẑ 漏进微调（运行前的修正）。
  R2 的常数输入（训练均值 z）也同样交叉拟合。
- **三档微调**（每档都在每个外层折的训练人上做；20% 的人做内层验证、按 LSD 早停；损失 = bin 1–85、仰角 ≥ −45° 的 dB MSE）：
  - **A（主分析）**：只调 SONICOM 实测那一行 FiLM：out' = e^δ·(out − β) + β + δβ，δ、δβ 逐频率（bin 0 不训练）。全批量 Adam。
  - **B（次要）**：A + 解码器所有 bias 向量（BitFit，约 9.9 万个）。
  - **D（次要）**：A + 整个解码器的权重（约 5050 万个），另加向预训练权重的 L2-SP 约束。
  - B、D 用小批量（8 只耳）、分组学习率（δ/δβ 1e-2，bias 1e-3，D 的权重 1e-5），每个 epoch 看一次内层验证，恢复最好的状态。
- **行**：R0 训练均值 HRTF；R0z 训练均值 z（冻结）；R1 几何 ẑ（冻结，= FFM_3_8）；R2 常数 + 微调；R3 几何 + 微调；
  R4 本人仿真 + 微调（只在 A 里做）。
- **关卡**：R0、R1、仿真行复现 FFM_3_8 开发集（LSD ≥ −45° 4.558、4.378、4.316 dB）。
- **主统计量**（每档各算一次，A 是主分析）：Δ = [LSD(R3) − LSD(R2)] − [LSD(R1) − LSD(R0)]，按人配对 bootstrap（2000）。
  R2 是关键对照：微调过的解码器只靠常数输入也能降低误差（学到更好的均值），直接拿 R3 和 R0 比会把这部分算到几何头上。
- **读法（已写死）**：|Δ| < 均值 LSD 的 1%（≈ 0.046 dB）或 CI 含 0 → 冻结的解码器不是瓶颈；
  Δ ≤ −0.046 dB 且 CI 在 0 以下 → 微调带来了额外的几何信息。R2 − R0、R4 只做描述。

## 运行

Colab GPU，`Run all`。A 很快；B、D 每一步都要对整个解码器反向传播：大显存 GPU（如 RTX PRO 6000 / A100）上合计约 30–60 分钟，
T4 会慢很多。显存不够时把 Cell 1 里 B、D 的 `batch_ears` 从 8 改成 4（只改批大小，不改其他设定），并在回报结果时说明。

## Drive 上要有的文件

与 FFM_3_8 相同（`ELEC5021/data_48k_2026-10-10.zip`、`ELEC5021/split_48k.json`、`HRTF_Checkpoints/FFM48k_measim_best.pth`、
`ELEC5020/SONICOM/mesh2z_samples.zip`、`HRTF_Checkpoints/mesh2z_sonicom_latents_v1.npz`、`HRTF_Checkpoints/mesh2z_rff_v1/oof_ridge.npz`），
另加 FFM_3_8 的 `HRTF_Checkpoints/mesh2z_48k_v1/dev_oof.npz`（关卡用）。结果写到 `HRTF_Checkpoints/mesh2z_48k_ft_v1/`。

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 0 — 路径 · 数据（与 FFM_3_8 相同；另读 FFM_3_8 保存的折外预测做关卡）
# ==========================================================================
import torch
import os, json, csv, time, hashlib, zipfile, shutil
import numpy as np
import healpy as hp
from scipy import stats
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
PATHS = dict(
    data_zip    = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip',
    split       = f'{DRIVE}/ELEC5021/split_48k.json',
    ckpt        = f'{SAVE_DIR}/FFM48k_measim_best.pth',
    samples_zip = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples.zip',
    subject_lists = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz',   # 只取受试者名单
    fold_ref      = f'{SAVE_DIR}/mesh2z_rff_v1/oof_ridge.npz',     # 只取折号
    ffm38_oof     = f'{SAVE_DIR}/mesh2z_48k_v1/dev_oof.npz',       # FFM_3_8 的折外 ẑ（关卡）
)
for k, p in PATHS.items():
    print(f'{k:14s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先把缺失的文件放好'
ZIP_SHA256 = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7'
SPLIT_SHA256 = 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561'
SAMPLES_SHA256 = '8340bdf73be1a8722015e877bcca2957570293484277989f1245eb8d65aa085a'
OUT = f'{SAVE_DIR}/mesh2z_48k_ft_v1'
os.makedirs(OUT, exist_ok=True)


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def unzip_checked(zip_path, sha, dest):
    stamp = f'{dest}/.zip_ok_{sha[:16]}'
    if os.path.exists(stamp):
        return
    t0 = time.time()
    assert sha256(zip_path) == sha, f'{zip_path}: SHA-256 不符'
    shutil.rmtree(dest, ignore_errors=True); os.makedirs(dest)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    open(stamp, 'w').write(sha)
    print(f'{os.path.basename(zip_path)} 核对并解压 ({time.time() - t0:.0f}s)')


unzip_checked(PATHS['data_zip'], ZIP_SHA256, '/content/data_48k_local')
unzip_checked(PATHS['samples_zip'], SAMPLES_SHA256, '/content/mesh2z_samples_local')
DATA, SAMPLES = '/content/data_48k_local/data_48k', '/content/mesh2z_samples_local/mesh2z_samples'
assert os.path.exists(f'{SAMPLES}/manifest.csv')
assert sha256(PATHS['split']) == SPLIT_SHA256, 'split_48k.json 的 SHA-256 不符'
SPLIT = json.load(open(PATHS['split']))
MIRROR = np.load(f'{DATA}/healpix_mirror_idx.npy')
assert np.array_equal(MIRROR[MIRROR], np.arange(3072))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_grad_enabled(False)          # 全局关梯度；只有微调函数内部打开
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ==========================================================================
# Cell 1 — 设定与登记（与 PROJECT_LOG §2C 2026-10-16 及其 Amendment 相同；跑之前不改）
# ==========================================================================
N_BINS, FS = 86, 48000
FREQS = np.arange(N_BINS) * FS / 256
EVAL_BINS = slice(1, N_BINS)              # bin 1–85 = 187.5 Hz – 15.94 kHz，不含 DC
EL = 90 - np.degrees(hp.pix2ang(16, np.arange(3072))[0])
MEAS_DIRS = EL >= -45.0                   # 2652 个方向
ALPHA_REL = list(np.logspace(-4, 5, 37))
N_FOLDS, N_INNER, SEED = 10, 5, 0         # 与 FFM_3_2_1 / Task 1 / FFM_3_8 相同的折
N_PERM, N_BOOT, BS = 100, 2000, 16
SIGMAS, M_RFF = [0.005, 0.01, 0.02, 0.05, 0.1], 256
DM, DS = 2, 4                             # FFM48k_measim 的 FiLM 行：SONICOM 实测、SONICOM 仿真

FT = dict(
    inner_val_frac=0.2, crossfit_groups=5,
    A=dict(lr=1e-2, max_steps=3000, patience=200),                                    # 全批量，单位：步
    B=dict(lr_film=1e-2, lr=1e-3, batch_ears=8, max_epochs=100, patience=10),         # 单位：epoch
    D=dict(lr_film=1e-2, lr=1e-5, batch_ears=8, max_epochs=50, patience=5, l2sp=1e-3),
)
VARIANTS = {'A': ['R2', 'R3', 'R4'], 'B': ['R2', 'R3'], 'D': ['R2', 'R3']}

FFM38_DEV = {'training mean': 4.558416, 'geometry': 4.377719, 'simulation': 4.316483}   # FFM_3_8 开发集 LSD ≥ −45°（关卡）

REGISTRATION = dict(
    log='PROJECT_LOG §2C 2026-10-16 (registered before any code) and its amendment (before any run)',
    question='does fine-tuning the decoder on the prediction task (loss on measured HRTF) change the geometry result?',
    encoder='FFM48k_measim (frozen; z targets as in FFM_3_8)',
    subjects='164 SONICOM development subjects, FFM_3_2_1 folds; development only (locked 36 spent 10-12)',
    inputs='held-out rows: kernel ridge fitted on the fold\'s training subjects (FFM_3_8); training rows: cross-fitted within '
           'the training subjects (5 groups, the fold\'s alpha); constant input (R2) cross-fitted likewise',
    variants={'A (primary)': 'decoder FiLM row of measured SONICOM: per-bin gamma and beta, out\' = exp(d)*(out - beta) + beta + db',
              'B (secondary)': 'A + all decoder bias vectors (BitFit)',
              'D (secondary)': 'A + all decoder weights, L2-SP towards the pretrained weights'},
    training='each outer fold, training subjects only; 20% of them as inner validation, early stopping by LSD; '
             'loss = MSE (dB) over bins 1-85, elevation >= -45 deg; settings in FT',
    rows={'R0': 'training-mean HRTF', 'R0z': 'training-mean z, frozen decoder', 'R1': 'geometry z-hat, frozen (FFM_3_8)',
          'R2': 'constant z, fine-tuned', 'R3': 'geometry z-hat, fine-tuned', 'R4 (A only)': 'own-simulation z-hat, fine-tuned'},
    gate='R0, R1 and the simulation row reproduce FFM_3_8 development LSD >= -45 deg (4.558, 4.378, 4.316 dB)',
    primary='per variant, A primary: Delta = [LSD(R3) - LSD(R2)] - [LSD(R1) - LSD(R0)], per subject, paired bootstrap 2000',
    reading='|Delta| < 1% of the mean LSD (~0.046 dB) or CI includes 0: the frozen decoder is not the limit; '
            'Delta <= -0.046 dB with CI below 0: fine-tuning adds geometry information. R2 - R0 and R4 descriptive.',
    not_done='per-subject (test-time) fine-tune; encoder fine-tune',
    ft=FT,
)
print(json.dumps(REGISTRATION, indent=1, ensure_ascii=False))

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（GELU + FiLM）
#
# 与 FFM48k_Train.ipynb 的模型定义逐字相同（构建时复制进来）。
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 载入 FFM48k_measim
# ==========================================================================
ck = torch.load(PATHS['ckpt'], map_location='cpu', weights_only=False)
cfg = ck['config']
assert ck['run_name'] == 'FFM48k_measim' and cfg['num_bins'] == N_BINS and cfg['num_datasets'] == 6
assert cfg['data_zip_sha256'] == ZIP_SHA256 and cfg['split_sha256'] == SPLIT_SHA256
model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32, num_datasets=6,
                             num_bins=N_BINS, tie_encoder_film=True).to(device)
model.load_state_dict(ck['model_state_dict']); model.eval()
FILM_ID = {d['tag']: d['id'] for d in ck['datasets']}
assert FILM_ID['SONICOM'] == DM and FILM_ID['SONICOM_SIM'] == DS
print(f"FFM48k_measim epoch {ck['epoch'] + 1} | val {ck['val_rmse']:.4f} dB | FiLM 行 {FILM_ID}")
del ck

In [ ]:
# ==========================================================================
# Cell 4 — 回归引擎：核 ridge 的对偶形式（Task 1 Cell 4），kernel 由函数按训练行生成
#
# Kfun(ref) 返回全部行的 kernel，但标准化、常数列筛选和 kernel 定标只用 ref 行（训练折）估计。
# 复现关卡用 Kfun = lambda ref: K_global（Task 1 的做法：在全部开发行上估计）。
# ==========================================================================
def subject_folds(subjects, k, rng):
    p = rng.permutation(subjects)
    return [np.sort(a) for a in np.array_split(p, min(k, len(p)))]


def make_plan(ns, seed=SEED):
    # FFM_3_2_1 的 build_plan：随机数的调用顺序相同，所以折相同
    rng = np.random.default_rng(seed)
    subj = np.repeat(np.arange(ns), 2)
    rows = lambda s: np.flatnonzero(np.isin(subj, s))
    plan = []
    for te in subject_folds(np.arange(ns), N_FOLDS, rng):
        tr = np.setdiff1d(np.arange(ns), te)
        inner = [(rows(np.setdiff1d(tr, v)), rows(v)) for v in subject_folds(tr, N_INNER, rng)]
        plan.append(((rows(tr), rows(te)), inner))
    return plan


def centred_pair(G, tr, va):
    Gtt, Gvt = G[np.ix_(tr, tr)], G[np.ix_(va, tr)]
    cm, tot = Gtt.mean(0), Gtt.mean()
    return Gtt - cm[None] - cm[:, None] + tot, Gvt - cm[None] - Gvt.mean(1)[:, None] + tot


def centred_vv_diag(G, tr, va):
    r = G[np.ix_(va, tr)].mean(1)
    return np.diag(G[np.ix_(va, va)]) - 2 * r + G[np.ix_(tr, tr)].mean()


def weights(K, tr, va, alphas):
    Ktt, Kvt = centred_pair(K, tr, va)
    lam, Q = np.linalg.eigh(Ktt)
    lam = np.clip(lam, 0, None)
    KvQ, scale = Kvt @ Q, np.trace(Ktt) / len(tr)
    return [(KvQ / (lam + a * scale)) @ Q.T for a in alphas]


def row_sse(W, G, tr, va):
    CYtt, CYvt = centred_pair(G, tr, va)
    return centred_vv_diag(G, tr, va) - 2 * np.einsum('ij,ij->i', W, CYvt) + np.einsum('ij,ij->i', W @ CYtt, W)


def run_cv(Kfun, Y, plan, keep_pred=True):
    G = Y @ Y.T
    n = len(Y)
    sse, sst = np.zeros(n), np.zeros(n)
    pred = np.zeros_like(Y) if keep_pred else None
    chosen = []
    for (tr, va), inner in plan:
        tot = np.zeros(len(ALPHA_REL))
        for itr, iva in inner:
            for j, W in enumerate(weights(Kfun(itr), itr, iva, ALPHA_REL)):
                tot[j] += row_sse(W, G, itr, iva).sum()
        j = int(np.argmin(tot)); chosen.append(j)
        W = weights(Kfun(tr), tr, va, [ALPHA_REL[j]])[0]
        sse[va] = row_sse(W, G, tr, va); sst[va] = centred_vv_diag(G, tr, va)
        if keep_pred:
            my = Y[tr].mean(0)
            pred[va] = my + W @ (Y[tr] - my)
    return dict(r2=float(1 - sse.sum() / sst.sum()), sse=sse, sst=sst, alpha_idx=chosen, pred=pred)


def block_gram(X, ref):
    # 一个特征块：只用 ref 行估计均值、标准差和常数列；除以列数（多块拼接时每块权重相同）
    mu, sd = X[ref].mean(0), X[ref].std(0)
    keep = sd > 1e-12 * max(sd.max(), 1e-300)
    Xz = (X[:, keep] - mu[keep]) / sd[keep]
    return Xz @ Xz.T / Xz.shape[1]


def unit_scaled(K, ref):
    # kernel 定标：ref 行上中心化后的平均对角元 = 1（Task 1 的 unit_trace 只在 ref 上算）
    n = len(ref); C = np.eye(n) - 1.0 / n
    return K / (np.trace(C @ K[np.ix_(ref, ref)] @ C) / n)


def source_kernel(blocks, ref):
    return unit_scaled(sum(block_gram(X, ref) for X in blocks.values()), ref)


def kernel_fn(sources, fold_internal):
    if fold_internal:
        return lambda ref: sum(source_kernel(S, ref) for S in sources)
    allrows = np.arange(len(next(iter(sources[0].values()))))
    K = sum(source_kernel(S, allrows) for S in sources)
    return lambda ref: K


def fit_predict(Kfun, Y_dev, n_dev, n_all, seed=SEED):
    # 测试用：α 由开发集上的 5 折按人分组 CV 选，再用全部开发行拟合，预测其余行
    G = Y_dev @ Y_dev.T
    rng = np.random.default_rng(seed); subj = np.repeat(np.arange(n_dev // 2), 2)
    rows = lambda s: np.flatnonzero(np.isin(subj, s))
    tot = np.zeros(len(ALPHA_REL))
    for v in subject_folds(np.arange(n_dev // 2), N_INNER, rng):
        itr, iva = rows(np.setdiff1d(np.arange(n_dev // 2), v)), rows(v)
        for j, W in enumerate(weights(Kfun(itr), itr, iva, ALPHA_REL)):
            tot[j] += row_sse(W, G, itr, iva).sum()
    j = int(np.argmin(tot))
    dev, te = np.arange(n_dev), np.arange(n_dev, n_all)
    W = weights(Kfun(dev), dev, te, [ALPHA_REL[j]])[0]
    my = Y_dev.mean(0)
    return my + W @ (Y_dev - my), j


# 自检：随机特征的 CV-R² 应在 0 附近或为负；两种标准化都要过
_plan = make_plan(40)
_Y = np.random.default_rng(3).standard_normal((80, 50)); _S = [{'x': np.random.default_rng(4).standard_normal((80, 30))}]
for fi in (False, True):
    print(f"自检（fold_internal={fi}）：随机特征 CV-R² = {run_cv(kernel_fn(_S, fi), _Y, _plan, keep_pred=False)['r2']:+.4f}（应 ≤ ~0）")

In [ ]:
# ==========================================================================
# Cell 5 — 数据 · latent · 几何特征 · 指标
# ==========================================================================
def load_rows(tag, ids):
    # 每人两行（左耳, 镜像后的右耳）；每个数组按 manifest 核对 SHA-256
    d = f'{DATA}/{tag}_48k'
    man = {m['id']: m for m in json.load(open(f'{d}/manifest.json'))}
    X = np.empty((2 * len(ids), N_BINS, 3072), np.float32)
    for k, s in enumerate(ids):
        for side in ('left', 'right'):
            f = f'{d}/npy_{side}/{man[s][side]}'
            assert sha256(f) == man[s][f'{side}_sha256'], f
            a = np.load(f)[:N_BINS]
            X[2 * k + (side == 'right')] = a[:, MIRROR] if side == 'right' else a
    return X


def encode(X, did):
    Z = []
    for b in range(0, len(X), BS):
        x = torch.from_numpy(X[b:b + BS]).to(device)
        _, z = model(x, torch.full((len(x),), did, dtype=torch.long, device=device))
        Z.append(z.reshape(len(x), -1).double().cpu().numpy())
    return np.concatenate(Z)


def geometry_blocks(sids):
    # FFM_3_2_1 Cell 4/5（= Task 1 sonicom_geometry），只返回原始特征块，不做标准化
    man = {f"SON_{r['subject']}": r for r in csv.DictReader(open(f'{SAMPLES}/manifest.csv'))}
    flip = np.array([1.0, -1.0, 1.0])
    G0T = torch.as_tensor(np.random.default_rng(12345).standard_normal((M_RFF, 3)).T, dtype=torch.float32, device=device)

    def rff(P):
        base = 2 * np.pi * (torch.as_tensor(P, dtype=torch.float32, device=device) @ G0T)
        return {sg: torch.cat([torch.cos(sg * base).mean(0), torch.sin(sg * base).mean(0)]).double().cpu().numpy()
                for sg in SIGMAS}

    pct = lambda a, q: float(np.percentile(a, q))
    simple, ear = [], []
    for s in sids:
        d = np.load(f'{SAMPLES}/{s[4:]}.npz')
        head = d['head_pts'][:20000].astype(np.float64)
        head = head[head[:, 2] > -75.0]
        for side in (0, 1):
            if side == 0:
                pts, canal, area, hd = d['earL_pts'], d['ear_left_mm'].astype(np.float64), float(man[s]['crop_area_L_mm2']), head
            else:
                pts, canal, area, hd = d['earR_pts'], d['ear_right_mm'].astype(np.float64) * flip, float(man[s]['crop_area_R_mm2']), head * flip
            assert canal[1] > 0
            q = pts[:20000].astype(np.float64) - canal
            out = q[q[:, 1] > 3.0]
            simple.append([canal[0], canal[1], canal[2], pct(q[:, 1], 95),
                           pct(out[:, 2], 95) - pct(out[:, 2], 5), pct(out[:, 0], 95) - pct(out[:, 0], 5),
                           area, pct(hd[:, 0], 99.5) - pct(hd[:, 0], 0.5), pct(hd[:, 2], 99.9)])
            ear.append(rff(q))
    blocks = {'simple': np.array(simple)}
    for sg in SIGMAS:
        blocks[f'ear_s{sg}'] = np.stack([f[sg] for f in ear])
    return blocks


M45 = torch.as_tensor(MEAS_DIRS, device=device)


def metrics(pred, X):
    """pred(b0, b1) -> (b, 86, 3072) 张量；X 是实测（numpy）。返回逐耳的各项指标"""
    out = {k: [] for k in ('se_all', 'lsd_el45', 'lsd_full', 'mae_el45', 'mae_full', 'sd_bin_el45', 'sd_bin_full')}
    for b0 in range(0, len(X), BS):
        b1 = min(b0 + BS, len(X))
        e = (pred(b0, b1).float() - torch.from_numpy(X[b0:b1]).to(device)).double()
        out['se_all'].append((e ** 2).sum((1, 2)))                    # 旧口径：全部 bin、全部方向
        eb = e[:, EVAL_BINS]
        d = torch.sqrt((eb ** 2).mean(1))                             # 每个方向沿频率 RMS
        out['lsd_full'].append(d.mean(1)); out['lsd_el45'].append(d[:, M45].mean(1))
        a = eb.abs()
        out['mae_full'].append(a.mean((1, 2))); out['mae_el45'].append(a[:, :, M45].mean((1, 2)))
        out['sd_bin_el45'].append(torch.sqrt((eb[:, :, M45] ** 2).mean(2)))
        out['sd_bin_full'].append(torch.sqrt((eb ** 2).mean(2)))
    return {k: torch.cat(v).cpu().numpy() for k, v in out.items()}


def per_subject(m):
    ns = len(m['se_all']) // 2
    S = {k: m[k].reshape(ns, 2).mean(1) for k in ('lsd_el45', 'lsd_full', 'mae_el45', 'mae_full')}
    S['rmse_all'] = np.sqrt(m['se_all'].reshape(ns, 2).sum(1) / (2 * N_BINS * 3072))
    return S


def summary(m):
    S = per_subject(m)
    r = {k: float(v.mean()) for k, v in S.items() if k != 'rmse_all'}
    r.update({f'sd_{k}': float(S[k].std(ddof=1)) for k in ('lsd_el45', 'mae_el45')})
    r['pooled_rmse'] = float(np.sqrt(m['se_all'].sum() / (len(m['se_all']) * N_BINS * 3072)))
    r['curve_el45'] = m['sd_bin_el45'].mean(0).tolist(); r['curve_full'] = m['sd_bin_full'].mean(0).tolist()
    return r


def decoder_pred(Zrows, did):
    return lambda b0, b1: model.decoder(torch.from_numpy(Zrows[b0:b1].reshape(-1, N_BINS, 64)).float().to(device),
                                        torch.full((b1 - b0,), did, dtype=torch.long, device=device))


def ae_pred(X, did):
    return lambda b0, b1: model(torch.from_numpy(X[b0:b1]).to(device),
                                torch.full((b1 - b0,), did, dtype=torch.long, device=device))[0]


def paired(a, b, rng):
    """a − b over subjects（负 = a 更好）：均值、bootstrap 95% CI、单侧配对 t 检验 p（H1: a < b）"""
    d = a - b
    bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(N_BOOT)])
    return dict(diff=float(d.mean()), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)),
                t_p_one_sided=float(stats.ttest_rel(a, b, alternative='less').pvalue))


MASUYAMA = {'Avg. HRTF': dict(mae=3.78, lsd=5.05), 'Mesh2HRTF (raw simulation)': dict(mae=7.50, lsd=10.53),
            'S2RNF': dict(mae=3.60, lsd=4.90)}     # 他们的表 1（SONICOM 25 人测试集；"RMSE" 即 LSD，到 20 kHz、不含 DC、793 个测点）


def show(title, rows, mean_key='training mean'):
    print(f'\n{title}')
    print(f"{'':<44}{'R²(z)':>8}{'LSD≥−45°':>10}{'vs均值':>8}{'MAE≥−45°':>10}{'LSD全球':>9}{'vs均值':>8}{'旧pooled':>10}")
    m0 = rows[mean_key]
    for k, r in rows.items():
        rel = lambda key: f"{(m0[key] - r[key]) / m0[key]:+8.1%}"
        r2 = f"{r['r2']:+8.4f}" if r.get('r2') is not None else f"{'':>8}"
        print(f"{k:<44}{r2}{r['lsd_el45']:10.3f}{rel('lsd_el45')}{r['mae_el45']:10.3f}{r['lsd_full']:9.3f}{rel('lsd_full')}{r['pooled_rmse']:10.3f}")
    print('  参照（Masuyama 2026 表 1，口径不同，只比"相对均值"）：' +
          '；'.join(f"{k} LSD {v['lsd']:.2f} / MAE {v['mae']:.2f}" for k, v in MASUYAMA.items()) +
          f"；S2RNF 相对均值 LSD {(5.05 - 4.90) / 5.05:+.1%}，MAE {(3.78 - 3.60) / 3.78:+.1%}")

In [ ]:
# ==========================================================================
# Cell 6 — 开发集：受试者、折、数据、latent、几何
# ==========================================================================
L = np.load(PATHS['subject_lists'])          # 只用 eval_sids / locked_sids
DEV = [str(s) for s in L['eval_sids']]; LOCKED = [str(s) for s in L['locked_sids']]
assert len(DEV) == 164 and len(LOCKED) == 36 and not set(DEV) & set(LOCKED)
assert set(DEV) <= set(SPLIT['SONICOM']['train']) | set(SPLIT['SONICOM']['val'])
assert set(LOCKED) <= set(SPLIT['SONICOM']['test'])
O = np.load(PATHS['fold_ref'], allow_pickle=True)   # 只用 dev / fold_of_subject
assert [str(s) for s in O['dev']] == DEV
PLAN = make_plan(len(DEV))
fold = np.full(len(DEV), -1)
for f, ((tr, va), _) in enumerate(PLAN):
    fold[np.unique(va // 2)] = f
assert np.array_equal(fold, O['fold_of_subject']), '折与 FFM_3_2_1 不同'
FOLD_ROW = np.repeat(fold, 2)
print(f'开发集 {len(DEV)} 人 / {2 * len(DEV)} 只耳，外层折与 FFM_3_2_1 相同；锁定 {len(LOCKED)} 人（本格不读）')

t0 = time.time()
X_meas, X_sim = load_rows('SONICOM', DEV), load_rows('SONICOM_SIM', DEV)
Zm, Zs = encode(X_meas, DM), encode(X_sim, DS)
GEO = geometry_blocks(DEV)
print(f'数据、latent、几何 {time.time() - t0:.0f}s | z {Zm.shape} | 几何块 ' + ', '.join(f'{k} {v.shape[1]}' for k, v in GEO.items()))
SOURCES = {'geometry': GEO, 'simulation': {'z_sim': Zs}}

# 训练折均值（实测、仿真）：每折一个
MU_M = np.stack([X_meas[tr].astype(np.float64).mean(0) for (tr, _), _ in PLAN]).astype(np.float32)
MU_S = np.stack([X_sim[tr].astype(np.float64).mean(0) for (tr, _), _ in PLAN]).astype(np.float32)
fold_mean = lambda MU: (lambda b0, b1: torch.from_numpy(MU[FOLD_ROW[b0:b1]]).to(device))
REF_PRED = {
    'training mean': fold_mean(MU_M),
    'raw simulation': lambda b0, b1: torch.from_numpy(X_sim[b0:b1]).to(device),
    'simulated deviation + measured mean': lambda b0, b1: torch.from_numpy(X_sim[b0:b1] - MU_S[FOLD_ROW[b0:b1]] + MU_M[FOLD_ROW[b0:b1]]).to(device),
    'AE reconstruction (reference)': ae_pred(X_meas, DM),
}
REF = {k: metrics(p, X_meas) for k, p in REF_PRED.items()}

In [ ]:
# ==========================================================================
# Cell 7 — ẑ（每折按 FFM_3_8 重新拟合）· 与 FFM_3_8 保存的预测核对 · 冻结解码器的行 · 关卡
# ==========================================================================
t0 = time.time()
CV = {name: run_cv(kernel_fn([SOURCES[name]], fold_internal=True), Zm, PLAN) for name in ('geometry', 'simulation')}
print(f'交叉验证 {time.time() - t0:.0f}s')
O38 = np.load(PATHS['ffm38_oof'])
assert [str(s) for s in O38['dev']] == DEV and np.array_equal(O38['fold'], fold)
for name in CV:
    ref = O38[f'zpred_{name}'].astype(np.float64)
    rel = np.abs(CV[name]['pred'] - ref).max() / np.abs(ref).max()
    print(f"  {name:<12} CV-R² {CV[name]['r2']:+.4f} | 与 FFM_3_8 dev_oof 的最大相对差 {rel:.1e}")
    assert rel < 1e-4, f'{name}: ẑ 没有复现 FFM_3_8 —— 停下来查'

ZBAR = np.stack([Zm[tr].mean(0) for (tr, _), _ in PLAN])        # 每折的训练均值 z
INPUTS = {'R0z / R2': ZBAR[FOLD_ROW], 'R1 / R3': CV['geometry']['pred'], 'sim / R4': CV['simulation']['pred']}
FROZEN = {'R0': REF['training mean'],
          'R0z': metrics(decoder_pred(INPUTS['R0z / R2'], DM), X_meas),
          'R1': metrics(decoder_pred(INPUTS['R1 / R3'], DM), X_meas),
          'sim frozen': metrics(decoder_pred(INPUTS['sim / R4'], DM), X_meas)}
for k, ref_key in (('R0', 'training mean'), ('R1', 'geometry'), ('sim frozen', 'simulation')):
    got = summary(FROZEN[k])['lsd_el45']
    print(f'  关卡 {k:<11} LSD≥−45° {got:.6f}（FFM_3_8 {FFM38_DEV[ref_key]:.6f}）')
    assert abs(got - FFM38_DEV[ref_key]) < 1e-3, f'{k} 没有复现 FFM_3_8'
print(f"  R0z（训练均值 z 解码）LSD≥−45° {summary(FROZEN['R0z'])['lsd_el45']:.4f}")
print('✓ 关卡通过')

In [ ]:
# ==========================================================================
# Cell 8 — 每个外层折：训练行的 ẑ 在训练人内部交叉拟合 → 三档微调（A / B / D）→ 预测该折的测试行
#
# 解码器输出已含 SONICOM 行的 FiLM：out = raw·γ + β。这一行 γ → γ·e^δ、β → β + δβ 等价于
#   out' = e^δ·(out − β) + β + δβ        （δ、δβ 逐频率，初值 0；bin 0 不在损失里，保持 0）
# B、D 在此之外再放开解码器的 bias（B）或全部权重（D）；其他数据库的 FiLM 行和编码器始终冻结。
# ==========================================================================
import copy
BETA = model.decoder.film.params(DM)[1].squeeze(-1).detach()            # (86,) SONICOM 行当前的有效 β
M45 = torch.as_tensor(MEAS_DIRS, device=device)
XS = torch.from_numpy(X_meas).to(device)[:, 1:][:, :, M45]              # (328, 85, 2652) 实测，训练区域
KFUN = {'geometry': kernel_fn([SOURCES['geometry']], True), 'simulation': kernel_fn([SOURCES['simulation']], True)}
SRC = {'R2': 'constant', 'R3': 'geometry', 'R4': 'simulation'}


def film_apply(out, d, db, beta):
    # out (b, F, n)；d、db 为 (F,) 或 (b, F)
    if d.dim() == 1:
        d, db = d[None], db[None]
    return torch.exp(d)[:, :, None] * (out - beta[None, :, None]) + beta[None, :, None] + db[:, :, None]


def ear_lsd(pred, X):
    return torch.sqrt(((pred - X) ** 2).mean(1)).mean(1)                # 每只耳的 LSD（该区域）


def ridge_predict(Kfun, fit_rows, pred_rows, j):
    W = weights(Kfun(fit_rows), fit_rows, pred_rows, [ALPHA_REL[j]])[0]
    my = Zm[fit_rows].mean(0)
    return my + W @ (Zm[fit_rows] - my)


def crossfit(src, tr, j, rng):
    """训练行的 ẑ：只用该折训练人，分 5 组，每组由其余 4 组拟合的模型预测（不含该折测试人的任何信息）"""
    Z = np.zeros((len(tr), Zm.shape[1]))
    for g in subject_folds(np.unique(tr // 2), FT['crossfit_groups'], rng):
        inside = np.isin(tr // 2, g)
        fit, pr = tr[~inside], tr[inside]
        Z[inside] = Zm[fit].mean(0) if src == 'constant' else ridge_predict(KFUN[src], fit, pr, j)
    return Z


def split_inner(tr, rng):
    """该折训练人里 20% 的人做内层验证；返回在 tr 里的位置"""
    subj = np.unique(tr // 2); perm = rng.permutation(subj)
    val = set(perm[:int(round(FT['inner_val_frac'] * len(subj)))].tolist())
    iv = np.array([k for k, r in enumerate(tr) if r // 2 in val]); it = np.array([k for k, r in enumerate(tr) if r // 2 not in val])
    return it, iv


def decode_region(dec, Z, bs=BS):
    out = torch.empty((len(Z), N_BINS - 1, int(MEAS_DIRS.sum())), device=device)
    for b0 in range(0, len(Z), bs):
        z = torch.as_tensor(Z[b0:b0 + bs], dtype=torch.float32, device=device).reshape(-1, N_BINS, 64)
        out[b0:b0 + len(z)] = dec(z, torch.full((len(z),), DM, dtype=torch.long, device=device))[:, 1:][:, :, M45]
    return out


def predict_full(dec, Z, d, db, bs=BS):
    """测试行的完整预测（86 bin × 3072 方向），numpy float32"""
    dd = torch.zeros(N_BINS, device=device); dbb = torch.zeros(N_BINS, device=device); dd[1:], dbb[1:] = d, db
    P = np.zeros((len(Z), N_BINS, 3072), np.float32)
    for b0 in range(0, len(Z), bs):
        z = torch.as_tensor(Z[b0:b0 + bs], dtype=torch.float32, device=device).reshape(-1, N_BINS, 64)
        out = dec(z, torch.full((len(z),), DM, dtype=torch.long, device=device))
        P[b0:b0 + len(z)] = film_apply(out, dd, dbb, BETA).cpu().numpy()
    return P


def finetune_A(OUT, X, it, iv):
    """A：全批量，只有 δ、δβ。OUT、X 是训练行在训练区域上的（解码输出、实测）"""
    cfg, beta = FT['A'], BETA[1:]
    d = torch.zeros(N_BINS - 1, device=device, requires_grad=True)
    db = torch.zeros(N_BINS - 1, device=device, requires_grad=True)
    opt = torch.optim.Adam([d, db], lr=cfg['lr'])
    v0 = float(ear_lsd(film_apply(OUT[iv], d.detach(), db.detach(), beta), X[iv]).mean())
    best = dict(v=v0, step=0, d=d.detach().clone(), db=db.detach().clone())
    with torch.enable_grad():
        for step in range(1, cfg['max_steps'] + 1):
            opt.zero_grad()
            loss = ((film_apply(OUT[it], d, db, beta) - X[it]) ** 2).mean()
            loss.backward(); opt.step()
            with torch.no_grad():
                v = float(ear_lsd(film_apply(OUT[iv], d, db, beta), X[iv]).mean())
            if v < best['v'] - 1e-6:
                best = dict(v=v, step=step, d=d.detach().clone(), db=db.detach().clone())
            elif step - best['step'] >= cfg['patience']:
                break
    diag = dict(inner_val_lsd_before=v0, inner_val_lsd_best=best['v'], best=best['step'], last=step,
                still_improving=step == cfg['max_steps'] and best['step'] > cfg['max_steps'] - cfg['patience'])
    return model.decoder, best['d'], best['db'], diag


def finetune_net(variant, Z, X, it, iv, seed):
    """B / D：小批量；δ、δβ 加上解码器的 bias（B）或全部权重（D）；返回微调后的解码器副本"""
    cfg, beta = FT[variant], BETA[1:]
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    dec = copy.deepcopy(model.decoder)
    for p in dec.parameters():
        p.requires_grad_(False)
    named = [(n, p) for n, p in dec.named_parameters() if not n.startswith('film.')]
    ps = [p for n, p in named if n.endswith('bias')] if variant == 'B' else [p for n, p in named]
    for p in ps:
        p.requires_grad_(True)
    d = torch.zeros(N_BINS - 1, device=device, requires_grad=True)
    db = torch.zeros(N_BINS - 1, device=device, requires_grad=True)
    opt = torch.optim.Adam([{'params': [d, db], 'lr': cfg['lr_film']}, {'params': ps, 'lr': cfg['lr']}])
    theta0 = [p.detach().clone() for p in ps] if variant == 'D' else None
    Zt = torch.as_tensor(Z, dtype=torch.float32, device=device).reshape(-1, N_BINS, 64)
    ids = lambda k: torch.full((k,), DM, dtype=torch.long, device=device)
    region = lambda zb: film_apply(dec(zb, ids(len(zb)))[:, 1:][:, :, M45], d, db, beta)

    def val_lsd():
        with torch.no_grad():
            return float(sum(ear_lsd(region(Zt[iv[b:b + cfg['batch_ears']]]), X[iv[b:b + cfg['batch_ears']]]).sum()
                             for b in range(0, len(iv), cfg['batch_ears'])) / len(iv))

    keep = lambda: dict(state={k: v.detach().clone() for k, v in dec.state_dict().items()}, d=d.detach().clone(), db=db.detach().clone())
    v0 = val_lsd(); best = dict(v=v0, epoch=0, **keep())
    for ep in range(1, cfg['max_epochs'] + 1):
        order = rng.permutation(it)
        with torch.enable_grad():
            for b in range(0, len(order), cfg['batch_ears']):
                k = order[b:b + cfg['batch_ears']]
                opt.zero_grad()
                loss = ((region(Zt[k]) - X[k]) ** 2).mean()
                if theta0 is not None:
                    loss = loss + cfg['l2sp'] * sum(((p - p0) ** 2).sum() for p, p0 in zip(ps, theta0))
                loss.backward(); opt.step()
        v = val_lsd()
        if v < best['v'] - 1e-6:
            best = dict(v=v, epoch=ep, **keep())
        elif ep - best['epoch'] >= cfg['patience']:
            break
    dec.load_state_dict(best['state']); dec.eval()
    for p in dec.parameters():
        p.requires_grad_(False)
    diag = dict(inner_val_lsd_before=v0, inner_val_lsd_best=best['v'], best=best['epoch'], last=ep, n_trainable=int(sum(p.numel() for p in ps)) + 2 * (N_BINS - 1),
                still_improving=ep == cfg['max_epochs'] and best['epoch'] > cfg['max_epochs'] - cfg['patience'])
    return dec, best['d'], best['db'], diag


KEYS = [f'{v}:{r}' for v, rows in VARIANTS.items() for r in rows]
PRED = {k: np.zeros(X_meas.shape, np.float32) for k in KEYS}
DIAG = {k: [] for k in KEYS}
FILM = {k: np.zeros((N_FOLDS, 2, N_BINS)) for k in KEYS}
XFIT = {r: [] for r in SRC}                         # 交叉拟合输入在训练行上的 R²(z)（描述）
t_all = time.time()
for f, ((tr, va), _) in enumerate(PLAN):
    t0 = time.time()
    it, iv = split_inner(tr, np.random.default_rng(SEED + 10 + f))          # 同一折里所有行、所有档共用
    Xr = XS[tr]
    for row, src in SRC.items():
        rows_v = [v for v in VARIANTS if row in VARIANTS[v]]
        if not rows_v:
            continue
        j = None if src == 'constant' else CV[src]['alpha_idx'][f]
        Z_tr = crossfit(src, tr, j, np.random.default_rng(SEED + 100 + f))   # 同一折里三种输入用同样的 5 组
        Z_va = np.broadcast_to(Zm[tr].mean(0), (len(va), Zm.shape[1])).copy() if src == 'constant' else CV[src]['pred'][va]
        XFIT[row].append(float(1 - ((Z_tr - Zm[tr]) ** 2).sum() / ((Zm[tr] - Zm[tr].mean(0)) ** 2).sum()))
        for v in rows_v:
            key = f'{v}:{row}'
            if v == 'A':
                OUTR = decode_region(model.decoder, Z_tr)
                dec, d, db, diag = finetune_A(OUTR, Xr, it, iv)
                del OUTR
            else:
                dec, d, db, diag = finetune_net(v, Z_tr, Xr, it, iv, seed=SEED + 1000 * (v == 'D') + 10 * f + list(SRC).index(row))
            PRED[key][va] = predict_full(dec, Z_va, d, db)
            DIAG[key].append(diag)
            FILM[key][f, 0, 1:], FILM[key][f, 1, 1:] = d.cpu().numpy(), db.cpu().numpy()
            if v != 'A':
                del dec; torch.cuda.empty_cache()
    print(f'  折 {f + 1:2d}/{N_FOLDS}  {time.time() - t0:.0f}s  ' +
          ' | '.join(f"{k} {DIAG[k][-1]['inner_val_lsd_before']:.3f}→{DIAG[k][-1]['inner_val_lsd_best']:.3f} @{DIAG[k][-1]['best']}" for k in KEYS), flush=True)
print(f'微调合计 {time.time() - t_all:.0f}s')
print('交叉拟合的训练输入 R²(z)（训练行上的平均；应与 FFM_3_8 的折外 R² 同量级，常数输入 ≈ 0）：' +
      ', '.join(f'{r} {np.mean(x):+.4f}' for r, x in XFIT.items()))
for k in KEYS:
    if any(x['still_improving'] for x in DIAG[k]):
        print(f'  ★ {k}：有折在最大步数 / epoch 附近仍在改善')
print('可训练参数：A 170；' + '；'.join(f"{v} {DIAG[f'{v}:R3'][0]['n_trainable']:,}" for v in ('B', 'D') if f'{v}:R3' in DIAG))

In [ ]:
# ==========================================================================
# Cell 9 — 结果：各行 · 每档的 Δ 与登记的读法（A 为主）· 描述性比较 · 图 · 保存
# ==========================================================================
import matplotlib.pyplot as plt
pmet = lambda P: metrics(lambda b0, b1: torch.from_numpy(P[b0:b1]).to(device), X_meas)
TUNED = {k: pmet(P) for k, P in PRED.items()}
ALLM = {**FROZEN, **TUNED}
LABEL = {'R0': 'R0 training-mean HRTF', 'R0z': 'R0z training-mean z, frozen', 'R1': 'R1 geometry, frozen (FFM_3_8)',
         'sim frozen': 'simulation, frozen (FFM_3_8)'}
LABEL.update({k: f"{k.split(':')[0]} · {dict(R2='R2 constant z', R3='R3 geometry', R4='R4 simulation')[k.split(':')[1]]}, fine-tuned" for k in KEYS})
ROWS = {LABEL[k]: dict(r2=None, **summary(ALLM[k])) for k in ['R0', 'R0z', 'R1', 'sim frozen'] + KEYS}
show(f'开发集（{len(DEV)} 人，折外；A = FiLM 行，B = + bias，D = + 全部解码器权重）', ROWS, mean_key=LABEL['R0'])

SUB = {k: per_subject(m)['lsd_el45'] for k, m in ALLM.items()}
rng = np.random.default_rng(SEED + 3)
THR = float(0.01 * SUB['R0'].mean())


def delta_stat(v):
    d = (SUB[f'{v}:R3'] - SUB[f'{v}:R2']) - (SUB['R1'] - SUB['R0'])
    bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(N_BOOT)])
    r = dict(delta=float(d.mean()), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)),
             t_p_two_sided=float(stats.ttest_1samp(d, 0).pvalue),
             geometry_contribution_frozen=float((SUB['R1'] - SUB['R0']).mean()),
             geometry_contribution_finetuned=float((SUB[f'{v}:R3'] - SUB[f'{v}:R2']).mean()), threshold=THR)
    if abs(r['delta']) < THR or r['lo'] <= 0 <= r['hi']:
        r['verdict'] = 'the frozen decoder is not the limit (|Delta| < threshold or CI includes 0)'
    elif r['delta'] <= -THR and r['hi'] < 0:
        r['verdict'] = 'fine-tuning adds geometry information: the frozen decoder understates the geometry route'
    else:
        r['verdict'] = 'the geometry contribution shrinks with fine-tuning (Delta > threshold, CI above 0): the frozen decoder does not limit it'
    return r


DELTA = {v: delta_stat(v) for v in VARIANTS}
print(f'\nΔ = [R3 − R2] − [R1 − R0]（LSD ≥ −45°，按人；负 = 微调后几何贡献更大）；阈值 ±{THR:.4f} dB')
for v, r in DELTA.items():
    print(f"  {v}{'（主）' if v == 'A' else '（次要）'}  几何贡献 冻结 {r['geometry_contribution_frozen']:+.4f} → 微调 {r['geometry_contribution_finetuned']:+.4f} dB"
          f" | Δ {r['delta']:+.4f} [{r['lo']:+.4f}, {r['hi']:+.4f}]，p = {r['t_p_two_sided']:.2g}\n      → {r['verdict']}")

SECONDARY = {}
for v in VARIANTS:
    SECONDARY[f'{v}: R2 − R0 (fine-tuning alone, constant input)'] = paired(SUB[f'{v}:R2'], SUB['R0'], rng)
    SECONDARY[f'{v}: R3 − R1 (fine-tuning on geometry)'] = paired(SUB[f'{v}:R3'], SUB['R1'], rng)
    SECONDARY[f'{v}: R3 − R0'] = paired(SUB[f'{v}:R3'], SUB['R0'], rng)
SECONDARY['A: R4 − simulation frozen'] = paired(SUB['A:R4'], SUB['sim frozen'], rng)
SECONDARY['A: R4 − R0'] = paired(SUB['A:R4'], SUB['R0'], rng)
SECONDARY['R0z − R0 (decoding the mean z)'] = paired(SUB['R0z'], SUB['R0'], rng)
print('\n描述性（负 = 前者更好；95% CI 按人 bootstrap）')
for k, r in SECONDARY.items():
    print(f"  {k:<48} {r['diff']:+.4f} dB [{r['lo']:+.4f}, {r['hi']:+.4f}]")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
fk = FREQS / 1000
for k, c in (('A:R2', 'k'), ('A:R3', 'C0'), ('A:R4', 'C1')):
    g, o = np.exp(FILM[k][:, 0, 1:]), FILM[k][:, 1, 1:]
    ax[0].plot(fk[1:], g.mean(0), c=c, label=k); ax[0].fill_between(fk[1:], g.min(0), g.max(0), color=c, alpha=.15)
    ax[1].plot(fk[1:], o.mean(0), c=c, label=k); ax[1].fill_between(fk[1:], o.min(0), o.max(0), color=c, alpha=.15)
ax[0].axhline(1, c='grey', lw=.6); ax[1].axhline(0, c='grey', lw=.6)
ax[0].set(xlabel='kHz', ylabel='gain e^δ', title='A: fine-tuned FiLM gain (mean, range over folds)')
ax[1].set(xlabel='kHz', ylabel='δβ (dB)', title='A: fine-tuned FiLM offset')
for k, c, ls in (('R0', 'k', '-'), ('R1', 'C0', '-'), ('A:R3', 'C0', '--'), ('B:R3', 'C2', '--'), ('D:R3', 'C3', '--'), ('D:R2', 'C7', ':')):
    if k in ALLM:
        ax[2].plot(fk[EVAL_BINS], summary(ALLM[k])['curve_el45'], c=c, ls=ls, lw=1.4, label=LABEL[k])
ax[2].set(xlabel='kHz', ylabel='RMS error over directions (dB)', title='Error vs frequency, elevation ≥ −45°')
for a in ax:
    a.grid(alpha=.3); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT}/ft_results.png', dpi=140); plt.show()

RES = dict(created=time.strftime('%Y-%m-%d %H:%M:%S'), notebook='mesh2z/FFM_3_9_DecoderFineTune.ipynb', registration=REGISTRATION,
           n_subjects=len(DEV), gate=dict(ffm38=FFM38_DEV, r2_cv={k: CV[k]['r2'] for k in CV}),
           crossfit_train_r2={r: x for r, x in XFIT.items()}, rows=ROWS, delta=DELTA, primary=DELTA['A'],
           secondary=SECONDARY, diagnostics=DIAG)
json.dump(RES, open(f'{OUT}/ft_results.json', 'w'), indent=1, default=float, ensure_ascii=False)
np.savez_compressed(f'{OUT}/ft_params.npz', freqs=FREQS, **{f"film_{k.replace(':', '_')}": v for k, v in FILM.items()},
                    **{f"subject_lsd_el45_{k.replace(':', '_').replace(' ', '_')}": v for k, v in SUB.items()})
print(f'\n-> {OUT}/ft_results.json, ft_params.npz, ft_results.png')

## 跑完之后

把 Cell 7–9 的输出贴回来（或下载 `mesh2z_48k_ft_v1/ft_results.json` 和 `ft_results.png` 到 `hrtf-GitHub/results/mesh2z_48k_ft_v1/`）。
先看：关卡；交叉拟合的训练输入 R²（几何应与 FFM_3_8 的 0.069 同量级）；每档每折的最佳步 / epoch 和 ★ 提示；
A 的 Δ 和读法（主分析），再看 B、D 的 Δ；R2 − R0（只靠微调、常数输入能得到多少）。